In [2]:
import os
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import nltk
from nltk.translate.gleu_score import sentence_gleu
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [3]:
train_df = pd.read_csv("/content/sample_data/train_clean.csv")
dev_df = pd.read_csv("/content/sample_data/dev_clean.csv")

train_dataset = Dataset.from_pandas(train_df[['input_text', 'target_text']])
dev_dataset = Dataset.from_pandas(dev_df[['input_text', 'target_text']])

In [4]:
MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

PREFIX = "fix grammar: "

def preprocess_function(examples):
    inputs = [PREFIX + str(text) for text in examples["input_text"]]
    targets = [str(text) for text in examples["target_text"]]

    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding=False)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding=False)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
dev_tokenized = dev_dataset.map(preprocess_function, batched=True, remove_columns=dev_dataset.column_names)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Map:   0%|          | 0/34308 [00:00<?, ? examples/s]

Map:   0%|          | 0/4384 [00:00<?, ? examples/s]

In [ ]:
from difflib import SequenceMatcher
import numpy as np

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.asarray(preds)
    preds = np.nan_to_num(preds, nan=tokenizer.pad_token_id, posinf=tokenizer.pad_token_id, neginf=tokenizer.pad_token_id)
    preds = np.clip(preds, 0, tokenizer.vocab_size - 1).astype(np.int64)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.nan_to_num(labels, nan=tokenizer.pad_token_id).astype(np.int64)
    labels = np.clip(labels, 0, tokenizer.vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    exact_matches = sum(p == l for p, l in zip(decoded_preds, decoded_labels))
    exact_match_acc = exact_matches / len(decoded_preds)

    sim_scores = []
    gleu_scores = []

    for pred, label in zip(decoded_preds, decoded_labels):
        pred_words = pred.split()
        label_words = label.split()

        matcher = SequenceMatcher(None, pred_words, label_words)
        sim_scores.append(matcher.ratio())

        score = sentence_gleu([label_words], pred_words)
        gleu_scores.append(score)

    word_similarity_acc = np.mean(sim_scores) * 100
    mean_gleu = np.mean(gleu_scores) * 100

    return {
        "exact_match_accuracy": round(exact_match_acc * 100, 2),
        "word_token_accuracy": round(word_similarity_acc, 2),
        "gleu_score": round(mean_gleu, 2)
    }

In [ ]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = Seq2SeqTrainingArguments(
    output_dir="./gec_flan_t5_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,          
    per_device_eval_batch_size=8,           
    gradient_accumulation_steps=2,          
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=128,            
    fp16=use_fp16,
    bf16=use_bf16,
    load_best_model_at_end=True,
    metric_for_best_model="gleu_score",
    greater_is_better=True,
    logging_steps=100
)

In [7]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=dev_tokenized,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics
)

In [8]:
trainer.train()
print("\nFinal Evaluation:", trainer.evaluate())

Epoch,Training Loss,Validation Loss,Exact Match Accuracy,Word Token Accuracy,Gleu Score
1,0.604841,0.254967,32.190000,92.360000,81.030000
2,0.507576,0.240819,34.990000,92.810000,82.120000
3,0.468939,0.246023,34.810000,92.810000,82.120000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Training Loss,Validation Loss,Epoch,Exact Match Accuracy,Word Token Accuracy,Gleu Score
0.468939,0.240819,3,34.990000,92.810000,82.120000



Final Evaluation: {'eval_loss': 0.240819051861763, 'eval_exact_match_accuracy': 34.99, 'eval_word_token_accuracy': 92.81, 'eval_gleu_score': 82.12}


In [9]:
SAVE_PATH = "./gec_flan_t5_saved_model"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print("Model saved to Colab session environment!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to Colab session environment!


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(SAVE_PATH)

def correct_text(input_str):
    inputs = tokenizer(f"fix grammar: {input_str}", return_tensors="pt")
    outputs = model.generate(**inputs, max_length=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_sentence = "I do done the task."
print("Original:", test_sentence)
print("Corrected:", correct_text(test_sentence))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Original: I do done the task.
Corrected: I have done the task.
